### Human in the loop

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from typing import Annotated
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

from langchain_tavily import TavilySearch
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.runnables import RunnableConfig

memory = MemorySaver()

llm = ChatGoogleGenerativeAI(model="gemini-3.7-flash")

class State(TypedDict):
    messages: Annotated[list, add_messages]

@tool
def human_assistance(query):
    """this is a tool for human assistance when assistance is needed"""
    # Implement the logic for human assistance here
    human_response = interrupt({"query": query})
    return human_response["data"]


tool= TavilySearch(max_results=2)

tools = [tool, human_assistance]
llm_with_tools = llm.bind_tools(tools)


def chatbbot(state: State):
    return {"messages":[llm_with_tools.invoke(state["messages"])]}

tool_node = ToolNode(tools=tools)

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

graph = graph_builder.compile(checkpointer=memory)

graph

In [ ]:
config: RunnableConfig = {"configurable": {"thread_id": "th1"}}
user_input = "Hello, I need assistance from human AI approval."

for event in graph.stream({"messages": user_input}, config=config, stream_mode="values"):
    print(event["messages"][-1])


In [ ]:
human_response = ("yes go ahead with the approval and proceed with getting current MSFT stock prioce of latest")

human_command = Command(resume={"data": human_response})
events = graph.stream(human_command, config=config, stream_mode="values")
for event in events:
    print(event["messages"][-1])
